In [1]:
import tensorflow as tf
import numpy as np
import tqdm
from tqdm import tqdm
import config

2026-02-10 14:38:48.931512: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 14:38:48.938018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770734328.946087 2778657 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770734328.948538 2778657 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-10 14:38:48.956872: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
gpus = tf.config.list_physical_devices(device_type='GPU')
print(gpus)
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.set_visible_devices(gpus[0], 'GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
  except RuntimeError as e:
    print(e)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2 Physical GPUs, 1 Logical GPUs


I0000 00:00:1770734331.286604 2778657 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5495 MB memory:  -> device: 0, name: NVIDIA RTX A4000, pci bus id: 0000:01:00.0, compute capability: 8.6


In [3]:
"""
  Evaluate a given quantized model on a given quantized validation set.
  Eventually save the accuracy of the model in the specified file.

  Args:
    model: tf.lite.Interpreter
      The model to be evaluated.
    valid_X: numpy.ndarray
      The quantized validation set to be used for evaluation.
    valid_Y: numpy.ndarray
      The labels of the validation set.
    model_name: str ["AlexNet", "ResNet50", "ResNet152", "VGG16"]
      The name of the model to be evaluated.
    quantized_type: str ["int8", "uint8", "int16"]
      The quantization type of the model to be evaluated.
"""
def evaulate_quantized_model_on_quantized_ds(model: tf.lite.Interpreter, valid_X: np.ndarray, valid_Y: np.ndarray, model_name: str, quantized_type: str):
  input_details = model.get_input_details()
  output_details = model.get_output_details()

  predictions = []
  desc = model_name + " " + quantized_type + " evaluation"

  for i in tqdm(range(len(valid_X)), desc=desc):
    model.set_tensor(input_details[0]['index'], valid_X[i:i+1])
    model.invoke()
    output_data = model.get_tensor(output_details[0]['index'])
    predictions.append(np.argmax(output_data, axis=1)[0]) 
  predictions = np.array(predictions)
  accuracy = (np.sum(predictions == valid_Y.squeeze())/len(predictions))*100
  
  with open(config.MODELS_ACCURACY_PATH, 'a') as file:
    file.write(model_name + " " + quantized_type + " CIFAR-10[:" + str(len(valid_Y)) +  "]: " + str(accuracy) + "\n")


In [ ]:
def evaluate_uint8_cifar10_x_alexnet():
  model = tf.lite.Interpreter(model_path=config.ALEXNET_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.ALEXNET_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'ALEXNET', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_alexnet()

In [ ]:
def evaluate_int8_cifar10_x_alexnet():
  model = tf.lite.Interpreter(model_path=config.ALEXNET_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.ALEXNET_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'ALEXNET', 
                                           'int8')
  
evaluate_int8_cifar10_x_alexnet()

In [ ]:
def evaluate_uint8_cifar10_x_resnet50():
  model = tf.lite.Interpreter(model_path=config.RESNET50_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.RESNET50_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'RESNET50', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_resnet50()

In [ ]:
def evaluate_int8_cifar10_x_resnet50():
  model = tf.lite.Interpreter(model_path=config.RESNET50_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.RESNET50_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'RESNET50', 
                                           'int8')
  
evaluate_int8_cifar10_x_resnet50()

In [ ]:
def evaluate_uint8_cifar10_x_resnet152():
  model = tf.lite.Interpreter(model_path=config.RESNET152_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.RESNET152_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'RESNET152', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_resnet152()

In [ ]:
def evaluate_int8_cifar10_x_resnet152():
  model = tf.lite.Interpreter(model_path=config.RESNET152_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.RESNET152_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'RESNET152', 
                                           'int8')
  
evaluate_int8_cifar10_x_resnet152()

In [ ]:
def evaluate_uint8_cifar10_x_vgg16():
  model = tf.lite.Interpreter(model_path=config.VGG16_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.VGG16_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'VGG16', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_vgg16()

In [ ]:
def evaluate_int8_cifar10_x_vgg16():
  model = tf.lite.Interpreter(model_path=config.VGG16_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.VGG16_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'VGG16', 
                                           'int8')
  
evaluate_int8_cifar10_x_vgg16()

In [ ]:
def evaluate_uint8_cifar10_x_mobilenet():
  model = tf.lite.Interpreter(model_path=config.MOBILENET_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.MOBILENET_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'MOBILENET', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_mobilenet()

In [ ]:
def evaluate_int8_cifar10_x_mobilenet():
  model = tf.lite.Interpreter(model_path=config.MOBILENET_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.MOBILENET_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'MOBILENET', 
                                           'int8')
  
evaluate_int8_cifar10_x_mobilenet()

In [ ]:
def evaluate_uint8_cifar10_x_mobilenetV2():
  model = tf.lite.Interpreter(model_path=config.MOBILENETV2_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.MOBILENETV2_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'MOBILENETV2', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_mobilenetV2()

In [ ]:
def evaluate_int8_cifar10_x_mobilenetV2():
  model = tf.lite.Interpreter(model_path=config.MOBILENETV2_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.MOBILENETV2_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'MOBILENETV2', 
                                           'int8')
  
evaluate_int8_cifar10_x_mobilenetV2()

In [ ]:
def evaluate_uint8_cifar10_x_efficientnetB0():
  model = tf.lite.Interpreter(model_path=config.EFFICIENTNETB0_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.EFFICIENTNETB0_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'EFFICIENTNETB0', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_efficientnetB0()

In [ ]:
def evaluate_int8_cifar10_x_efficientnetB0():
  model = tf.lite.Interpreter(model_path=config.EFFICIENTNETB0_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.EFFICIENTNETB0_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'EFFICIENTNETB0', 
                                           'int8')
  
evaluate_int8_cifar10_x_efficientnetB0()

In [ ]:
def evaluate_uint8_cifar10_x_efficientnetB3():
  model = tf.lite.Interpreter(model_path=config.EFFICIENTNETB3_U8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.EFFICIENTNETB3_U8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'EFFICIENTNETB3', 
                                           'uint8')
  
evaluate_uint8_cifar10_x_efficientnetB3()

In [4]:
def evaluate_int8_cifar10_x_efficientnetB3():
  model = tf.lite.Interpreter(model_path=config.EFFICIENTNETB3_I8_MODEL_PATH)
  model.allocate_tensors()
  valid_X = np.load(config.EFFICIENTNETB3_I8_VALIDATION_SET_PREPROCESSED_PATH+".npy")
  valid_Y = np.load(config.VALIDATION_LABELS_PATH)
  evaulate_quantized_model_on_quantized_ds(model, 
                                           valid_X[config.N_IMAGES_SLICE], 
                                           valid_Y[config.N_IMAGES_SLICE],
                                           'EFFICIENTNETB3', 
                                           'int8')
  
evaluate_int8_cifar10_x_efficientnetB3()

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
EFFICIENTNETB3 int8 evaluation: 100%|██████████| 2000/2000 [00:25<00:00, 79.60it/s]
